## 一.指定超参数

In [1]:
# 1.相关库引入
import pandas as pd # 数据处理库
from sklearn.model_selection import train_test_split # 划分训练集和测试集
from sklearn.compose import ColumnTransformer # 列转换器
from sklearn.preprocessing import OneHotEncoder, StandardScaler # 独热编码和标准化
from sklearn.neighbors import KNeighborsClassifier # KNN分类器
import joblib # 保存模型

In [2]:
# 2.加载数据
heart_disease = pd.read_csv('2.测试数据/data/heart_disease.csv')
print(heart_disease.info())
print(heart_disease.nunique())
heart_disease

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   年龄        1025 non-null   int64  
 1   性别        1025 non-null   int64  
 2   胸痛类型      1025 non-null   int64  
 3   静息血压      1025 non-null   int64  
 4   胆固醇       1025 non-null   int64  
 5   空腹血糖      1025 non-null   int64  
 6   静息心电图结果   1025 non-null   int64  
 7   最大心率      1025 non-null   int64  
 8   运动性心绞痛    1025 non-null   int64  
 9   运动后的ST下降  1025 non-null   float64
 10  峰值ST段的斜率  1025 non-null   int64  
 11  主血管数量     1025 non-null   int64  
 12  地中海贫血     1025 non-null   int64  
 13  是否患有心脏病   1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB
None
年龄           41
性别            2
胸痛类型          4
静息血压         49
胆固醇         152
空腹血糖          2
静息心电图结果       3
最大心率         91
运动性心绞痛        2
运动后的ST下降     40
峰值ST段的斜率      3
主血管数量         5
地中海贫血         4
是否患

,年龄,性别,胸痛类型,静息血压,胆固醇,空腹血糖,静息心电图结果,最大心率,运动性心绞痛,运动后的ST下降,峰值ST段的斜率,主血管数量,地中海贫血,是否患有心脏病
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [3]:
# 3.数据集划分
X = heart_disease.drop('是否患有心脏病', axis=1)
y = heart_disease['是否患有心脏病']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(717, 13) (308, 13) (717,) (308,)


In [4]:
# 4.特征工程
"""
这里列按类型分组，目的是对症下药，连续数值做缩放，离散类别做编码，二元保持不变
"""
# 数值型特征
numerical_features = ["年龄", "静息血压", "胆固醇", "最大心率", "运动后的ST下降", "主血管数量"]
# 类别型特征
categorical_features = ["胸痛类型", "静息心电图结果", "峰值ST段的斜率", "地中海贫血"]
# 二元特征
binary_features = ["性别", "空腹血糖", "运动性心绞痛"]

# 创建一个列转换器
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features), # 数值列 → 标准化(均值0方差1)
        ("cat", OneHotEncoder(drop="first"), categorical_features),  # 类别列 → 独热编码(丢第一列避免多重共线)
        ("bin", "passthrough", binary_features) # 二元列 → 原样透传
    ]
)

# 特征转换
"""
fit_transform：计算训练集的均值/方差、类别字典，并完成转换
transform：用训练时学到的参数去变换测试集
"""
X_train_transformed = column_transformer.fit_transform(X_train)
X_test_transformed = column_transformer.transform(X_test)
print(X_train_transformed.shape,X_test_transformed.shape)

(717, 19) (308, 19)


In [6]:
# 5.模型训练
knn = KNeighborsClassifier(n_neighbors=3)

knn.fit(X_train_transformed, y_train)
"""
这里会输入KNN的一个参数面板：
1.n_neighbors=3：最近3个近邻投票
2.weights='uniform'：每个邻居权重相同
3.algorithm='auto'：邻居搜索算法选择，这里是自动挑选
4.leaf_size=30：KD/Ball Tree 的叶子大小（结点中样本数上限）
5.p=2：使用欧式距离
6.metric='minkowski'：闵可夫斯基距离
7.metric_params=None：某些距离的额外参数设置
8.n_jobs=None：并行度，-1表示用所有CPU

备注：其中p=2在metric='minkowski'的基础上才能用，具体问AI
"""

"\n这里会输入KNN的一个参数面板：\n1.n_neighbors=3：最近3个近邻投票\n2.weights='uniform'：每个邻居权重相同\n3.algorithm='auto'：邻居搜索算法选择，这里是自动挑选\n4.leaf_size=30：KD/Ball Tree 的叶子大小（结点中样本数上限）\n5.p=2：使用欧式距离\n6.metric='minkowski'：闵可夫斯基距离\n7.metric_params=None：某些距离的额外参数设置\n8.n_jobs=None：并行度，-1表示用所有CPU\n\n备注：其中p=2在metric='minkowski'的基础上才能用，具体问AI\n"

In [7]:
# 6.模型评估
score = knn.score(X_test_transformed, y_test)
print(score)

0.9253246753246753


In [22]:
# 7.模型保存
joblib.dump(knn, 'models/knn_model')

['models/knn_model']

In [35]:
# 8.模型加载，对新数据做预测
knn_loaded = joblib.load('models/knn_model')
"""
这里不能是[10]，这是单个列表，相当于一维，需要是[10:11]的矩阵，相当于二维
"""
y_pred = knn_loaded.predict(X_test_transformed[10:11])
print(f"预测类别：{y_pred}，真实类别：{y_test[10]}")

预测类别：[1]，真实类别：1


## 二.网格搜索型

In [8]:
# 1.相关库引入
import pandas as pd # 数据处理库
from sklearn.model_selection import train_test_split,GridSearchCV # 划分训练集和测试集；网格搜索
from sklearn.compose import ColumnTransformer # 列转换器
from sklearn.preprocessing import OneHotEncoder, StandardScaler # 独热编码和标准化
from sklearn.neighbors import KNeighborsClassifier # KNN分类器
import joblib # 保存模型

In [9]:
# 2.加载数据
heart_disease = pd.read_csv('2.测试数据/data/heart_disease.csv')
print(heart_disease.info())
print(heart_disease.nunique())
heart_disease

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   年龄        1025 non-null   int64  
 1   性别        1025 non-null   int64  
 2   胸痛类型      1025 non-null   int64  
 3   静息血压      1025 non-null   int64  
 4   胆固醇       1025 non-null   int64  
 5   空腹血糖      1025 non-null   int64  
 6   静息心电图结果   1025 non-null   int64  
 7   最大心率      1025 non-null   int64  
 8   运动性心绞痛    1025 non-null   int64  
 9   运动后的ST下降  1025 non-null   float64
 10  峰值ST段的斜率  1025 non-null   int64  
 11  主血管数量     1025 non-null   int64  
 12  地中海贫血     1025 non-null   int64  
 13  是否患有心脏病   1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB
None
年龄           41
性别            2
胸痛类型          4
静息血压         49
胆固醇         152
空腹血糖          2
静息心电图结果       3
最大心率         91
运动性心绞痛        2
运动后的ST下降     40
峰值ST段的斜率      3
主血管数量         5
地中海贫血         4
是否患

,年龄,性别,胸痛类型,静息血压,胆固醇,空腹血糖,静息心电图结果,最大心率,运动性心绞痛,运动后的ST下降,峰值ST段的斜率,主血管数量,地中海贫血,是否患有心脏病
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [10]:
# 3.数据集划分
X = heart_disease.drop('是否患有心脏病', axis=1)
y = heart_disease['是否患有心脏病']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(717, 13) (308, 13) (717,) (308,)


In [11]:
# 4.特征工程
"""
这里列按类型分组，目的是对症下药，连续数值做缩放，离散类别做编码，二元保持不变
"""
# 数值型特征
numerical_features = ["年龄", "静息血压", "胆固醇", "最大心率", "运动后的ST下降", "主血管数量"]
# 类别型特征
categorical_features = ["胸痛类型", "静息心电图结果", "峰值ST段的斜率", "地中海贫血"]
# 二元特征
binary_features = ["性别", "空腹血糖", "运动性心绞痛"]

# 创建一个列转换器
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features), # 数值列 → 标准化(均值0方差1)
        ("cat", OneHotEncoder(drop="first"), categorical_features),  # 类别列 → 独热编码(丢第一列避免多重共线)
        ("bin", "passthrough", binary_features) # 二元列 → 原样透传
    ]
)

# 特征转换
"""
fit_transform：计算训练集的均值/方差、类别字典，并完成转换
transform：用训练时学到的参数去变换测试集
"""
X_train_transformed = column_transformer.fit_transform(X_train)
X_test_transformed = column_transformer.transform(X_test)
print(X_train_transformed.shape,X_test_transformed.shape)

(717, 19) (308, 19)


In [13]:
# 5.创建模型
knn = KNeighborsClassifier()

# 定义网格搜索参数列表
param_grad = {"n_neighbors": list(range(1, 11)), "weights": ["uniform", "distance"]}

grid_search_cv = GridSearchCV(knn, param_grad, cv=10) # 10折交叉验证

In [14]:
# 6.模型训练
grid_search_cv.fit(X_train_transformed, y_train)

,estimator,KNeighborsClassifier()
,param_grid,"{'n_neighbors': [1, 2, ...], 'weights': ['uniform', 'distance']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,9


In [16]:
# 7.模型评估
results = pd.DataFrame(grid_search_cv.cv_results_)
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000638,0.000088,0.013234,0.000751,1,uniform,"{'n_neighbors': 1, 'weights': 'uniform'}",0.986111,1.000000,0.972222,0.986111,1.000000,0.986111,0.972222,0.985915,0.957746,0.943662,0.979010,0.016953,9
1,0.000926,0.000207,0.013691,0.002239,1,distance,"{'n_neighbors': 1, 'weights': 'distance'}",0.986111,1.000000,0.972222,0.986111,1.000000,0.986111,0.972222,0.985915,0.957746,0.943662,0.979010,0.016953,9
2,0.000634,0.000168,0.010820,0.001074,2,uniform,"{'n_neighbors': 2, 'weights': 'uniform'}",0.944444,0.930556,0.916667,0.930556,0.958333,0.944444,0.902778,0.943662,0.901408,0.845070,0.921792,0.031117,12
3,0.000693,0.000210,0.012876,0.001110,2,distance,"{'n_neighbors': 2, 'weights': 'distance'}",0.986111,1.000000,0.972222,0.986111,1.000000,0.986111,0.972222,0.985915,0.957746,0.943662,0.979010,0.016953,9
4,0.000633,0.000178,0.010899,0.000843,3,uniform,"{'n_neighbors': 3, 'weights': 'uniform'}",0.930556,0.833333,0.902778,0.861111,0.875000,0.916667,0.888889,0.845070,0.845070,0.774648,0.867312,0.043574,17
5,0.000689,0.000177,0.012057,0.000849,3,distance,"{'n_neighbors': 3, 'weights': 'distance'}",0.986111,1.000000,0.972222,1.000000,1.000000,0.986111,0.972222,1.000000,0.971831,0.943662,0.983216,0.017548,5
6,0.000589,0.000037,0.010612,0.001064,4,uniform,"{'n_neighbors': 4, 'weights': 'uniform'}",0.902778,0.805556,0.888889,0.875000,0.888889,0.875000,0.875000,0.859155,0.816901,0.774648,0.856182,0.040169,20
7,0.000751,0.000194,0.012577,0.001081,4,distance,"{'n_neighbors': 4, 'weights': 'distance'}",0.986111,1.000000,0.972222,1.000000,1.000000,0.986111,0.972222,0.985915,0.971831,0.943662,0.981808,0.016689,6
8,0.000636,0.000079,0.011000,0.000915,5,uniform,"{'n_neighbors': 5, 'weights': 'uniform'}",0.930556,0.805556,0.847222,0.916667,0.888889,0.902778,0.888889,0.901408,0.802817,0.760563,0.864534,0.054298,18
9,0.000779,0.000190,0.012419,0.000981,5,distance,"{'n_neighbors': 5, 'weights': 'distance'}",0.986111,0.986111,0.972222,1.000000,1.000000,0.986111,0.972222,0.985915,0.957746,0.957746,0.980419,0.014341,8


In [17]:
# 8.获取最佳模型
print(grid_search_cv.best_estimator_) # 最佳模型
print(grid_search_cv.best_score_) # 最佳得分
print(grid_search_cv.best_params_) # 最佳参数

KNeighborsClassifier(n_neighbors=9, weights='distance')
0.9887910798122066
{'n_neighbors': 9, 'weights': 'distance'}


In [18]:
# 9.使用最佳模型做测试
knn = grid_search_cv.best_estimator_
print(knn.score(X_test_transformed, y_test)) # 模型准确率

1.0
